In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Tuple, Union

# Optional: only needed if you want pretty string output
METRICS = ["MAE", "RAE", "R2", "Spearman R", "Kendall's Tau"]


def clip_and_log_transform(y: np.ndarray):
    y = np.clip(y, a_min=0, a_max=None)
    return np.log10(y + 1)


def bootstrap_sampling(size: int, n_samples: int) -> np.ndarray:
    rng = np.random.default_rng(0)
    return rng.choice(size, size=(n_samples, size), replace=True)


def metrics_per_ep(pred: np.ndarray, true: np.ndarray) -> Tuple[float, float, float, float, float]:
    from scipy.stats import spearmanr, kendalltau
    from sklearn.metrics import mean_absolute_error, r2_score

    mae = mean_absolute_error(true, pred)
    denom = np.mean(np.abs(true - np.mean(true)))
    rae = mae / denom if denom != 0 else np.nan

    if np.nanstd(true) == 0:
        r2 = np.nan
    else:
        r2 = r2_score(true, pred)

    spr = spearmanr(true, pred).statistic
    ktau = kendalltau(true, pred).statistic

    return mae, rae, r2, spr, ktau


def bootstrap_metrics(
    pred: np.ndarray,
    true: np.ndarray,
    endpoint: str,
    n_bootstrap_samples: int = 1000
) -> pd.DataFrame:
    cols = ["Sample", "Endpoint", "Metric", "Value"]
    bootstrap_results = pd.DataFrame(columns=cols)

    for i, indx in enumerate(bootstrap_sampling(true.shape[0], n_bootstrap_samples)):
        mae, rae, r2, spr, ktau = metrics_per_ep(pred[indx], true[indx])

        scores = pd.DataFrame(
            [
                [i, endpoint, "MAE", mae],
                [i, endpoint, "RAE", rae],
                [i, endpoint, "R2", r2],
                [i, endpoint, "Spearman R", spr],
                [i, endpoint, "Kendall's Tau", ktau],
            ],
            columns=cols,
        )

        bootstrap_results = pd.concat([bootstrap_results, scores], ignore_index=True)

    return bootstrap_results


def map_metric_to_stats(df: pd.DataFrame) -> pd.DataFrame:
    cols_drop = []
    for col in METRICS:
        mean_col = f"mean_{col}"
        std_col = f"std_{col}"
        df[col] = df.apply(
            lambda row: f"{row[mean_col]:.2f} +/- {row[std_col]:.2f}",
            axis=1
        )
        cols_drop.extend([mean_col, std_col])
    return df.drop(columns=cols_drop)


def _load_df(obj: Union[str, Path, pd.DataFrame]) -> pd.DataFrame:
    if isinstance(obj, pd.DataFrame):
        return obj.copy()
    return pd.read_csv(obj)


def calculate_multi_file_metrics(
    prediction_files: dict,
    truth_file: Union[str, Path, pd.DataFrame],
    true_col: str,
    pred_col: str = None,
    id_col: str = "Molecule Name",
    n_bootstrap_samples: int = 1000,
    transform_predictions: bool = False,
    pretty: bool = False,
) -> pd.DataFrame:
    """
    Evaluate multiple prediction files against one truth file.

    Parameters
    ----------
    prediction_files : dict
        Mapping like:
        {
            "model_a": "preds_a.csv",
            "model_b": "preds_b.csv"
        }
        Values can be file paths or DataFrames.
    truth_file : str | Path | pd.DataFrame
        Ground-truth file/DataFrame.
    true_col : str
        Column in truth_file containing the true values.
    pred_col : str | None
        Prediction column name in each prediction file.
        If None, the function uses the single non-id column automatically.
    id_col : str
        Key used to align rows, e.g. "Molecule Name".
    transform_predictions : bool
        False = only true values are clip+log transformed.
        True = predictions are also clip+log transformed.
    pretty : bool
        If True, returns "mean +/- std" strings.

    Returns
    -------
    pd.DataFrame
        Ranked table, lowest mean_MAE first.
    """
    truth_df = _load_df(truth_file)

    required_truth = [id_col, true_col]
    missing_truth = [c for c in required_truth if c not in truth_df.columns]
    if missing_truth:
        raise ValueError(f"Truth file is missing required columns: {missing_truth}")

    if truth_df[id_col].duplicated().any():
        raise ValueError(f"Truth file contains duplicated {id_col} values.")

    truth_df = truth_df[[id_col, true_col]].copy()
    truth_df[true_col] = pd.to_numeric(truth_df[true_col], errors="coerce")

    all_results = []

    for model_name, pred_source in prediction_files.items():
        pred_df = _load_df(pred_source)

        if id_col not in pred_df.columns:
            raise ValueError(f"{model_name}: missing required column '{id_col}'")

        if pred_df[id_col].duplicated().any():
            raise ValueError(f"{model_name}: duplicated {id_col} values found")

        # Determine prediction column
        if pred_col is None:
            candidate_cols = [c for c in pred_df.columns if c != id_col]
            if len(candidate_cols) != 1:
                raise ValueError(
                    f"{model_name}: could not infer prediction column. "
                    f"Expected exactly one non-id column, found {candidate_cols}"
                )
            pred_col_used = candidate_cols[0]
        else:
            pred_col_used = pred_col
            if pred_col_used not in pred_df.columns:
                raise ValueError(f"{model_name}: missing prediction column '{pred_col_used}'")

        pred_df = pred_df[[id_col, pred_col_used]].copy()
        pred_df[pred_col_used] = pd.to_numeric(pred_df[pred_col_used], errors="coerce")

        # Check all truth molecules exist in predictions
        if not truth_df[id_col].isin(pred_df[id_col]).all():
            missing_ids = truth_df.loc[~truth_df[id_col].isin(pred_df[id_col]), id_col].tolist()
            raise ValueError(
                f"{model_name}: predictions file is missing {len(missing_ids)} molecules from truth file. "
                f"First few missing: {missing_ids[:5]}"
            )

        merged = truth_df.merge(pred_df, on=id_col, how="inner")
        merged = merged[[id_col, true_col, pred_col_used]].dropna()

        if merged.empty:
            raise ValueError(f"{model_name}: no valid rows left after numeric conversion and dropping NaNs")

        y_true = merged[true_col].to_numpy()
        y_pred = merged[pred_col_used].to_numpy()

        # Transform true values
        y_true_transformed = clip_and_log_transform(y_true)

        # Optional transform of predictions
        if transform_predictions:
            y_pred_transformed = clip_and_log_transform(y_pred)
        else:
            y_pred_transformed = y_pred

        bootstrap_df = bootstrap_metrics(
            y_pred_transformed,
            y_true_transformed,
            endpoint=model_name,
            n_bootstrap_samples=n_bootstrap_samples,
        )

        result = (
            bootstrap_df
            .pivot_table(
                index=["Endpoint"],
                columns="Metric",
                values="Value",
                aggfunc=["mean", "std"]
            )
            .reset_index()
        )

        result.columns = [f"{a}_{b}" if a != "" else b for a, b in result.columns]
        result = result.rename(columns={"Endpoint_": "Model"})
        all_results.append(result)

    final = pd.concat(all_results, ignore_index=True)

    ordered_cols = (
        ["Model"]
        + [f"mean_{m}" for m in METRICS]
        + [f"std_{m}" for m in METRICS]
    )
    final = final[ordered_cols]

    final = final.sort_values("mean_MAE", ascending=True).reset_index(drop=True)
    final.insert(0, "Rank", np.arange(1, len(final) + 1))

    if pretty:
        final = map_metric_to_stats(final)

    return final

In [2]:
def calculate_multi_df_metrics(
    dfs: dict,
    pred_col: str,
    true_col: str,
    id_col: str = None,
    n_bootstrap_samples: int = 1000,
    transform_predictions: bool = False,
    pretty: bool = False,
) -> pd.DataFrame:
    """
    Evaluate multiple dataframes, each containing:
      - pred_col
      - true_col
    and rank them by lowest mean_MAE.

    Parameters
    ----------
    dfs : dict
        Mapping like:
        {
            "model_a": df_a,
            "model_b": df_b,
            "model_c": df_c
        }
    pred_col : str
        Name of prediction column in each dataframe.
    true_col : str
        Name of true-value column in each dataframe.
    id_col : str | None
        Optional identifier column used to verify alignment across dataframes.
        If provided, truth values are checked by id.
    """
    if not dfs:
        raise ValueError("dfs is empty")

    model_names = list(dfs.keys())
    first_name = model_names[0]
    first_df = dfs[first_name].copy()

    required = [pred_col, true_col] + ([id_col] if id_col else [])
    missing = [c for c in required if c not in first_df.columns]
    if missing:
        raise ValueError(f"{first_name} is missing required columns: {missing}")

    # Build reference truth from the first dataframe
    ref_cols = [true_col] + ([id_col] if id_col else [])
    ref_truth = first_df[ref_cols].copy()
    ref_truth[true_col] = pd.to_numeric(ref_truth[true_col], errors="coerce")

    if id_col:
        if ref_truth[id_col].duplicated().any():
            raise ValueError(f"{first_name} has duplicated values in {id_col}")
        ref_truth = ref_truth.sort_values(id_col).reset_index(drop=True)
    else:
        ref_truth = ref_truth.reset_index(drop=True)

    all_results = []

    for model_name, df in dfs.items():
        work_df = df.copy()

        required = [pred_col, true_col] + ([id_col] if id_col else [])
        missing = [c for c in required if c not in work_df.columns]
        if missing:
            raise ValueError(f"{model_name} is missing required columns: {missing}")

        work_df[pred_col] = pd.to_numeric(work_df[pred_col], errors="coerce")
        work_df[true_col] = pd.to_numeric(work_df[true_col], errors="coerce")

        if id_col:
            if work_df[id_col].duplicated().any():
                raise ValueError(f"{model_name} has duplicated values in {id_col}")

            truth_check = work_df[[id_col, true_col]].sort_values(id_col).reset_index(drop=True)
            if not ref_truth[id_col].equals(truth_check[id_col]):
                raise ValueError(f"{model_name} does not have the same {id_col} values as {first_name}")

            # Check identical truth values
            if not np.allclose(
                ref_truth[true_col].to_numpy(dtype=float),
                truth_check[true_col].to_numpy(dtype=float),
                equal_nan=True
            ):
                raise ValueError(f"{model_name} does not have identical {true_col} values as {first_name}")

            eval_df = work_df[[id_col, pred_col, true_col]].dropna().sort_values(id_col).reset_index(drop=True)
        else:
            truth_check = work_df[[true_col]].reset_index(drop=True)

            if len(ref_truth) != len(truth_check):
                raise ValueError(f"{model_name} does not have the same number of rows as {first_name}")

            if not np.allclose(
                ref_truth[true_col].to_numpy(dtype=float),
                truth_check[true_col].to_numpy(dtype=float),
                equal_nan=True
            ):
                raise ValueError(f"{model_name} does not have identical {true_col} values as {first_name}")

            eval_df = work_df[[pred_col, true_col]].dropna().reset_index(drop=True)

        if eval_df.empty:
            raise ValueError(f"{model_name}: no valid rows left after dropping NaNs")

        y_pred = eval_df[pred_col].to_numpy()
        y_true = eval_df[true_col].to_numpy()

        # Transform true values
        y_true_transformed = clip_and_log_transform(y_true)

        # Optional: also transform predictions
        if transform_predictions:
            y_pred_transformed = clip_and_log_transform(y_pred)
        else:
            y_pred_transformed = y_pred

        bootstrap_df = bootstrap_metrics(
            y_pred_transformed,
            y_true_transformed,
            endpoint=model_name,
            n_bootstrap_samples=n_bootstrap_samples,
        )

        result = (
            bootstrap_df
            .pivot_table(
                index=["Endpoint"],
                columns="Metric",
                values="Value",
                aggfunc=["mean", "std"]
            )
            .reset_index()
        )

        result.columns = [f"{a}_{b}" if a != "" else b for a, b in result.columns]
        result = result.rename(columns={"Endpoint_": "Model"})
        all_results.append(result)

    final = pd.concat(all_results, ignore_index=True)

    ordered_cols = (
        ["Model"]
        + [f"mean_{m}" for m in METRICS]
        + [f"std_{m}" for m in METRICS]
    )
    final = final[ordered_cols]

    final = final.sort_values("mean_MAE", ascending=True).reset_index(drop=True)
    final.insert(0, "Rank", np.arange(1, len(final) + 1))

    if pretty:
        final = map_metric_to_stats(final)

    return final

In [3]:
df_expansion_chemprop = pd.read_csv("Predictions/expansion_chemprop_log10.csv")
df_openadmet_chemprop_high = pd.read_csv("Predictions/openadmet_chemprop_log10_high.csv")
df_openadmet_chemprop_low = pd.read_csv("Predictions/openadmet_chemprop_log10_low.csv")
df_no_chembl_chemprop_high = pd.read_csv("Predictions/no_chembl_chemprop_log10_high.csv")
df_no_chembl_chemprop_low = pd.read_csv("Predictions/no_chembl_chemprop_log10_low.csv")
df_no_scale_chemprop = pd.read_csv("Predictions/no_scale_chemprop_log10.csv")
df_all_chemprop_high = pd.read_csv("Predictions/all_chemprop_log10_high.csv")
df_all_chemprop_low = pd.read_csv("Predictions/all_chemprop_log10_low.csv")

In [4]:
dfs = {
    "Expansion_Chemprop": df_expansion_chemprop,
    "OpenADMET_Chemprop_high": df_openadmet_chemprop_high,
    "OpenADMET_Chemprop_low": df_openadmet_chemprop_low,
    "No_Chembl_Chemprop_high": df_no_chembl_chemprop_high,
    "No_Chembl_Chemprop_low": df_no_chembl_chemprop_low,
    "No_Scale_Chemprop": df_no_scale_chemprop,
    "All_Chemprop_high": df_all_chemprop_high,
    "All_Chemprop_low": df_all_chemprop_low,
}

ranked_df = calculate_multi_df_metrics(
    dfs=dfs,
    pred_col="y_pred_log10",
    true_col="hlm_value_true",
    id_col="Molecule Name",   # or None if rows are already in identical order
    n_bootstrap_samples=1000,
    transform_predictions=False,
    pretty=False
)

ranked_df

C:\Users\talag\AppData\Local\Temp\ipykernel_27756\1526751746.py:62: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  bootstrap_results = pd.concat([bootstrap_results, scores], ignore_index=True)
C:\Users\talag\AppData\Local\Temp\ipykernel_27756\1526751746.py:62: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  bootstrap_results = pd.concat([bootstrap_results, scores], ignore_index=True)
C:\Users\talag\AppData\Local\Temp\ipykernel_27756\1526751746.py:62: FutureWarning: The behavior of DataFrame concate

,Rank,Model,mean_MAE,mean_RAE,mean_R2,mean_Spearman R,mean_Kendall's Tau,std_MAE,std_RAE,std_R2,std_Spearman R,std_Kendall's Tau
0,1,OpenADMET_Chemprop_high,0.273260,0.745419,0.398271,0.642055,0.460783,0.008254,0.022695,0.031798,0.022619,0.018665
1,2,No_Chembl_Chemprop_low,0.276061,0.753066,0.390359,0.647582,0.464139,0.008358,0.023188,0.031997,0.022630,0.018818
2,3,No_Scale_Chemprop,0.282510,0.770602,0.357022,0.616833,0.436639,0.008433,0.021502,0.030354,0.023042,0.018482
3,4,Expansion_Chemprop,0.283682,0.773853,0.371258,0.639085,0.455168,0.008322,0.023046,0.032921,0.023049,0.018791
4,5,OpenADMET_Chemprop_low,0.284023,0.774789,0.363722,0.619481,0.443742,0.008282,0.023129,0.034804,0.025101,0.020107
5,6,No_Chembl_Chemprop_high,0.284037,0.774724,0.370978,0.626298,0.443450,0.008324,0.019520,0.025766,0.022606,0.018100
6,7,All_Chemprop_low,0.297509,0.811575,0.283938,0.587120,0.420168,0.009022,0.025041,0.040284,0.025989,0.020478
7,8,All_Chemprop_high,0.307389,0.838458,0.239032,0.562788,0.401628,0.009183,0.023117,0.037612,0.027420,0.020946
